In [1]:
# ============================================================
# CELL 1 - PROJECT TITLE
# ============================================================

print("=" * 90)
print("S3MambaDA VERSION 2")
print("Enhanced Subject-Independent Motor Imagery EEG Classification")
print("=" * 90)

print()
print("Major upgrades:")
print("1. Correct MNE event mapping")
print("2. Four-class validation")
print("3. Stable learnable Sinc filters")
print("4. Larger bidirectional temporal encoder")
print("5. Classification warm-up")
print("6. Gradual domain adaptation")
print("7. Reduced auxiliary-loss weights")
print("8. Training-only EEG augmentation")
print("9. Source validation split")
print("10. Best-checkpoint selection")
print("11. Adaptive Batch Normalization")
print("12. Paper-ready metrics and figures")

S3MambaDA VERSION 2
Enhanced Subject-Independent Motor Imagery EEG Classification

Major upgrades:
1. Correct MNE event mapping
2. Four-class validation
3. Stable learnable Sinc filters
4. Larger bidirectional temporal encoder
5. Classification warm-up
6. Gradual domain adaptation
7. Reduced auxiliary-loss weights
8. Training-only EEG augmentation
9. Source validation split
10. Best-checkpoint selection
11. Adaptive Batch Normalization
12. Paper-ready metrics and figures


In [2]:
# ============================================================
# CELL 2 - IMPORTS
# ============================================================

import os
import math
import random
import json
import time
import copy
import warnings

from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mne

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader, Subset

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    auc
)

from sklearn.manifold import TSNE

warnings.filterwarnings("ignore")
mne.set_log_level("ERROR")

print("Imports completed successfully.")

Imports completed successfully.


In [3]:
# ============================================================
# CELL 3 - EXPERIMENT CONFIGURATION
# ============================================================

SEED = 42

# ------------------------------------------------------------
# DATASET
# ------------------------------------------------------------

DATA_DIR = "./eegmmidb"

TOTAL_SUBJECTS = 109

RUNS = [4, 6, 8, 10, 12, 14]

# ------------------------------------------------------------
# TIME WINDOW
# ------------------------------------------------------------

TMIN = 0.5
TMAX = 3.5

FS = 250

N_SAMPLES = int(
    (TMAX - TMIN) * FS
)

# ------------------------------------------------------------
# CHANNEL CONFIGURATION
# ------------------------------------------------------------

# Options:
#
# "FIRST22"
# "ALL64"
# "CUSTOM"
#
CHANNEL_MODE = "FIRST22"

# Leave empty initially.
# After running the channel-name diagnostic cell,
# you can specify a motor-area montage here.
#
# Example format:
#
# CUSTOM_CHANNELS = [
#     "C3",
#     "Cz",
#     "C4",
# ]
#
CUSTOM_CHANNELS = []

if CHANNEL_MODE == "FIRST22":
    N_CHANNELS = 22

elif CHANNEL_MODE == "ALL64":
    N_CHANNELS = 64

elif CHANNEL_MODE == "CUSTOM":

    if len(CUSTOM_CHANNELS) == 0:
        raise ValueError(
            "CUSTOM_CHANNELS cannot be empty."
        )

    N_CHANNELS = len(
        CUSTOM_CHANNELS
    )

else:

    raise ValueError(
        "Invalid CHANNEL_MODE."
    )

# ------------------------------------------------------------
# CLASSES
# ------------------------------------------------------------

NUM_CLASSES = 4

CLASS_NAMES = [
    "Left Hand",
    "Right Hand",
    "Both Fists",
    "Both Feet"
]

# ------------------------------------------------------------
# SINC FILTER
# ------------------------------------------------------------

NUM_FILTERS = 10

SINC_KERNEL_SIZE = 81

SINC_MIN_FREQ = 1.0

SINC_MAX_FREQ = 40.0

# ------------------------------------------------------------
# SPATIAL
# ------------------------------------------------------------

SPATIAL_DIM = 64

# ------------------------------------------------------------
# TEMPORAL
# ------------------------------------------------------------

GRU_HIDDEN = 64

GRU_LAYERS = 2

GRU_DROPOUT = 0.20

# ------------------------------------------------------------
# TRAINING
# ------------------------------------------------------------

BATCH_SIZE = 64

EPOCHS = 200

WARMUP_EPOCHS = 20

LEARNING_RATE = 3e-4

WEIGHT_DECAY = 1e-4

LABEL_SMOOTHING = 0.05

# Maximum weights after warm-up
DOMAIN_WEIGHT_MAX = 0.15

SUPCON_WEIGHT_MAX = 0.10

SUPCON_TEMPERATURE = 0.07

GRADIENT_CLIP = 1.0

# ------------------------------------------------------------
# VALIDATION
# ------------------------------------------------------------

VALIDATION_SUBJECT_FRACTION = 0.10

# ------------------------------------------------------------
# EVALUATION
# ------------------------------------------------------------

# PROJECT
# RANDOM
# EXHAUSTIVE

EVAL_MODE = "PROJECT"

CURATED_TEST_POOL = [
    4,
    15,
    23,
    29,
    31,
    42,
    55,
    71,
    82,
    95
]

NUM_TEST_FOLDS = 10

NUM_TRAIN_SUBJECTS = 99

# ------------------------------------------------------------
# TRAIN AUGMENTATION
# ------------------------------------------------------------

USE_AUGMENTATION = True

AUG_NOISE_STD = 0.01

AMPLITUDE_MIN = 0.90

AMPLITUDE_MAX = 1.10

TEMPORAL_MASK_PROB = 0.30

CHANNEL_DROPOUT_PROB = 0.20

# ------------------------------------------------------------
# OUTPUT DIRECTORIES
# ------------------------------------------------------------

RESULTS_DIR = Path(
    "./S3MambaDA_V2_RESULTS"
)

FIGURES_DIR = (
    RESULTS_DIR / "FIGURES"
)

CHECKPOINT_DIR = (
    RESULTS_DIR / "CHECKPOINTS"
)

TABLES_DIR = (
    RESULTS_DIR / "TABLES"
)

for directory in [
    RESULTS_DIR,
    FIGURES_DIR,
    CHECKPOINT_DIR,
    TABLES_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

print("Configuration loaded.")

print()
print("Input:")
print(
    f"{N_CHANNELS} channels × {N_SAMPLES} samples"
)

print()
print("Training:")
print(
    f"{EPOCHS} epochs"
)

print(
    f"{WARMUP_EPOCHS} warm-up epochs"
)

print(
    f"Learning rate = {LEARNING_RATE}"
)

print()
print("Domain weight max:")
print(
    DOMAIN_WEIGHT_MAX
)

print(
    "SupCon weight max:",
    SUPCON_WEIGHT_MAX
)

Configuration loaded.

Input:
22 channels × 750 samples

Training:
200 epochs
20 warm-up epochs
Learning rate = 0.0003

Domain weight max:
0.15
SupCon weight max: 0.1


In [4]:
# ============================================================
# CELL 4 - REPRODUCIBILITY
# ============================================================

def seed_everything(
    seed=SEED
):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(seed)

    if torch.backends.cudnn.is_available():

        torch.backends.cudnn.deterministic = True

        torch.backends.cudnn.benchmark = False


seed_everything()

print(
    "Random seed:",
    SEED
)

Random seed: 42


In [5]:
# ============================================================
# CELL 5 - DEVICE
# ============================================================

if torch.cuda.is_available():

    DEVICE = torch.device(
        "cuda"
    )

elif torch.backends.mps.is_available():

    DEVICE = torch.device(
        "mps"
    )

else:

    DEVICE = torch.device(
        "cpu"
    )


print(
    "PyTorch:",
    torch.__version__
)

print(
    "Device:",
    DEVICE
)

if DEVICE.type == "mps":

    print(
        "Apple Silicon MPS detected."
    )

PyTorch: 2.10.0
Device: mps
Apple Silicon MPS detected.


In [6]:
# ============================================================
# CELL 6 - EEG CHANNEL NAME DIAGNOSTIC
# ============================================================

diagnostic_file = (
    Path(DATA_DIR)
    / "S004"
    / "S004R04.edf"
)

raw = mne.io.read_raw_edf(
    str(diagnostic_file),
    preload=False,
    verbose=False
)

print(
    "Total channels:",
    len(raw.ch_names)
)

print()

for index, channel in enumerate(
    raw.ch_names
):

    print(
        f"{index:02d}: {channel}"
    )

Total channels: 64

00: Fc5.
01: Fc3.
02: Fc1.
03: Fcz.
04: Fc2.
05: Fc4.
06: Fc6.
07: C5..
08: C3..
09: C1..
10: Cz..
11: C2..
12: C4..
13: C6..
14: Cp5.
15: Cp3.
16: Cp1.
17: Cpz.
18: Cp2.
19: Cp4.
20: Cp6.
21: Fp1.
22: Fpz.
23: Fp2.
24: Af7.
25: Af3.
26: Afz.
27: Af4.
28: Af8.
29: F7..
30: F5..
31: F3..
32: F1..
33: Fz..
34: F2..
35: F4..
36: F6..
37: F8..
38: Ft7.
39: Ft8.
40: T7..
41: T8..
42: T9..
43: T10.
44: Tp7.
45: Tp8.
46: P7..
47: P5..
48: P3..
49: P1..
50: Pz..
51: P2..
52: P4..
53: P6..
54: P8..
55: Po7.
56: Po3.
57: Poz.
58: Po4.
59: Po8.
60: O1..
61: Oz..
62: O2..
63: Iz..


In [7]:
# ============================================================
# CELL 7 - AVAILABLE SUBJECTS
# ============================================================

def get_available_subjects(
    data_dir,
    maximum=TOTAL_SUBJECTS
):

    subjects = []

    for subject in range(
        1,
        maximum + 1
    ):

        folder = (
            Path(data_dir)
            / f"S{subject:03d}"
        )

        if folder.exists():

            subjects.append(
                subject
            )

    return subjects


AVAILABLE_SUBJECTS = \
    get_available_subjects(
        DATA_DIR
    )

print(
    "Available subjects:",
    len(AVAILABLE_SUBJECTS)
)

print(
    AVAILABLE_SUBJECTS
)

Available subjects: 109
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109]


In [8]:
# ============================================================
# CELL 8 - CORRECTED EEGMMIDB DATASET
# ============================================================

class EEGMMIDB_Dataset(
    Dataset
):

    def __init__(
        self,
        data_dir,
        subjects,
        runs=RUNS,
        tmin=TMIN,
        tmax=TMAX,
        training=False
    ):

        self.data_dir = str(
            data_dir
        )

        self.subjects = list(
            subjects
        )

        self.runs = list(
            runs
        )

        self.tmin = tmin

        self.tmax = tmax

        self.training = training

        self.epochs = []

        self.labels = []

        self.subject_ids = []

        self.run_ids = []

        self.load_data()


    # ========================================================
    # CHANNEL SELECTION
    # ========================================================

    def select_channels(
        self,
        raw
    ):

        if CHANNEL_MODE == "FIRST22":

            raw.pick(
                raw.ch_names[:22]
            )

        elif CHANNEL_MODE == "ALL64":

            raw.pick(
                raw.ch_names[:64]
            )

        elif CHANNEL_MODE == "CUSTOM":

            missing = [

                ch

                for ch in CUSTOM_CHANNELS

                if ch not in raw.ch_names

            ]

            if missing:

                raise ValueError(

                    f"Missing channels: "
                    f"{missing}"

                )

            raw.pick(
                CUSTOM_CHANNELS
            )

        return raw


    # ========================================================
    # LOAD DATA
    # ========================================================

    def load_data(
        self
    ):

        for subject in self.subjects:

            subject_folder = (
                f"S{subject:03d}"
            )

            subject_path = (
                Path(self.data_dir)
                / subject_folder
            )


            if not subject_path.exists():

                print(
                    f"[WARNING] Missing "
                    f"{subject_folder}"
                )

                continue


            for run in self.runs:

                edf_file = (

                    subject_path
                    /
                    f"{subject_folder}R{run:02d}.edf"

                )


                if not edf_file.exists():

                    print(
                        f"[WARNING] Missing "
                        f"{edf_file}"
                    )

                    continue


                try:

                    # --------------------------------------------
                    # READ EDF
                    # --------------------------------------------

                    raw = mne.io.read_raw_edf(

                        str(edf_file),

                        preload=True,

                        verbose=False

                    )


                    # --------------------------------------------
                    # SELECT CHANNELS
                    # --------------------------------------------

                    raw = self.select_channels(
                        raw
                    )


                    # --------------------------------------------
                    # RESAMPLE
                    # --------------------------------------------

                    raw.resample(
                        FS,
                        verbose=False
                    )


                    # --------------------------------------------
                    # EVENTS
                    # --------------------------------------------

                    events, event_id = \
                        mne.events_from_annotations(

                            raw,

                            verbose=False

                        )


                    if (
                        "T1" not in event_id
                        or
                        "T2" not in event_id
                    ):

                        print(
                            f"[WARNING] "
                            f"T1/T2 missing "
                            f"in S{subject:03d} "
                            f"R{run:02d}"
                        )

                        continue


                    # --------------------------------------------
                    # REAL MNE EVENT IDS
                    #
                    # T1 normally = 2
                    # T2 normally = 3
                    # --------------------------------------------

                    extraction_mapping = {

                        "T1":
                            event_id["T1"],

                        "T2":
                            event_id["T2"]

                    }


                    # --------------------------------------------
                    # FINAL ML LABELS
                    # --------------------------------------------

                    if run in [
                        4,
                        8,
                        12
                    ]:

                        label_mapping = {

                            "T1": 0,

                            "T2": 1

                        }


                    elif run in [
                        6,
                        10,
                        14
                    ]:

                        label_mapping = {

                            "T1": 2,

                            "T2": 3

                        }

                    else:

                        continue


                    # --------------------------------------------
                    # EPOCH
                    # --------------------------------------------

                    epochs = mne.Epochs(

                        raw,

                        events,

                        event_id=
                            extraction_mapping,

                        tmin=self.tmin,

                        tmax=(
                            self.tmax
                            -
                            1.0 / FS
                        ),

                        baseline=None,

                        preload=True,

                        verbose=False

                    )


                    data = \
                        epochs.get_data()


                    event_codes = \
                        epochs.events[:, -1]


                    # --------------------------------------------
                    # CONVERT EVENTS TO ML LABELS
                    # --------------------------------------------

                    for index in range(
                        len(data)
                    ):

                        event_code = int(
                            event_codes[index]
                        )


                        if event_code == \
                           event_id["T1"]:

                            label = \
                                label_mapping["T1"]


                        elif event_code == \
                             event_id["T2"]:

                            label = \
                                label_mapping["T2"]


                        else:

                            continue


                        self.epochs.append(

                            data[index]
                            .astype(
                                np.float32
                            )

                        )


                        self.labels.append(
                            label
                        )


                        self.subject_ids.append(
                            subject - 1
                        )


                        self.run_ids.append(
                            run
                        )


                except Exception as error:

                    print(

                        f"[ERROR] "
                        f"S{subject:03d} "
                        f"R{run:02d} | "
                        f"{type(error).__name__}: "
                        f"{error}"

                    )


    # ========================================================
    # AUGMENTATION
    # ========================================================

    def augment(
        self,
        x
    ):

        if not self.training:
            return x


        if not USE_AUGMENTATION:
            return x


        # -----------------------------------------------------
        # Amplitude scaling
        # -----------------------------------------------------

        scale = torch.empty(
            1
        ).uniform_(
            AMPLITUDE_MIN,
            AMPLITUDE_MAX
        )

        x = x * scale


        # -----------------------------------------------------
        # Gaussian noise
        # -----------------------------------------------------

        noise = torch.randn_like(
            x
        ) * AUG_NOISE_STD

        x = x + noise


        # -----------------------------------------------------
        # Temporal masking
        # -----------------------------------------------------

        if (
            random.random()
            <
            TEMPORAL_MASK_PROB
        ):

            mask_length = random.randint(
                10,
                min(
                    40,
                    x.shape[1] // 5
                )
            )


            max_start = (
                x.shape[1]
                -
                mask_length
            )


            if max_start > 0:

                start = random.randint(
                    0,
                    max_start
                )


                x[
                    :,
                    start:
                    start + mask_length
                ] = 0


        # -----------------------------------------------------
        # Channel dropout
        # -----------------------------------------------------

        if (
            random.random()
            <
            CHANNEL_DROPOUT_PROB
        ):

            channel = random.randint(
                0,
                x.shape[0] - 1
            )


            x[channel] = 0


        return x


    # ========================================================
    # LENGTH
    # ========================================================

    def __len__(
        self
    ):

        return len(
            self.epochs
        )


    # ========================================================
    # ITEM
    # ========================================================

    def __getitem__(
        self,
        index
    ):

        x = torch.tensor(
            self.epochs[index],
            dtype=torch.float32
        )


        y = torch.tensor(
            self.labels[index],
            dtype=torch.long
        )


        subject = torch.tensor(
            self.subject_ids[index],
            dtype=torch.long
        )


        # -----------------------------------------------------
        # PER-TRIAL / PER-CHANNEL NORMALIZATION
        # -----------------------------------------------------

        mean = x.mean(
            dim=1,
            keepdim=True
        )

        std = x.std(
            dim=1,
            keepdim=True
        )


        x = (
            x - mean
        ) / (
            std + 1e-6
        )


        # -----------------------------------------------------
        # TRAINING AUGMENTATION
        # -----------------------------------------------------

        x = self.augment(
            x
        )


        return (
            x,
            y,
            subject
        )

In [9]:
# ============================================================
# CELL 9 - S004 DATASET VALIDATION
# ============================================================

s004_dataset = EEGMMIDB_Dataset(

    DATA_DIR,

    subjects=[4],

    training=False

)


print(
    "Total trials:",
    len(s004_dataset)
)


print(
    "Class distribution:",
    Counter(
        s004_dataset.labels
    )
)


print(
    "Run distribution:",
    Counter(
        s004_dataset.run_ids
    )
)


sample_x, \
sample_y, \
sample_subject = \
    s004_dataset[0]


print(
    "Trial shape:",
    sample_x.shape
)


print(
    "Expected:",
    (
        N_CHANNELS,
        N_SAMPLES
    )
)


assert len(
    s004_dataset
) > 0


assert set(
    s004_dataset.labels
) == {
    0,
    1,
    2,
    3
}


assert sample_x.shape == (
    N_CHANNELS,
    N_SAMPLES
)


print()
print(
    "✓ DATASET VALIDATION PASSED"
)

Total trials: 90
Class distribution: Counter({0: 23, 3: 23, 1: 22, 2: 22})
Run distribution: Counter({4: 15, 6: 15, 8: 15, 10: 15, 12: 15, 14: 15})
Trial shape: torch.Size([22, 750])
Expected: (22, 750)

✓ DATASET VALIDATION PASSED


In [10]:
# ============================================================
# CELL 10 - TARGET COHORT VALIDATION
# ============================================================

def validate_subject(
    subject
):

    dataset = EEGMMIDB_Dataset(

        DATA_DIR,

        subjects=[subject],

        training=False

    )


    distribution = Counter(
        dataset.labels
    )


    runs = Counter(
        dataset.run_ids
    )


    return {

        "subject":
            subject,

        "trials":
            len(dataset),

        "class_0":
            distribution.get(
                0,
                0
            ),

        "class_1":
            distribution.get(
                1,
                0
            ),

        "class_2":
            distribution.get(
                2,
                0
            ),

        "class_3":
            distribution.get(
                3,
                0
            ),

        "runs":
            str(dict(runs))

    }


target_subjects = [

    subject

    for subject in CURATED_TEST_POOL

    if subject in AVAILABLE_SUBJECTS

][
    :NUM_TEST_FOLDS
]


validation_table = pd.DataFrame(

    [
        validate_subject(
            subject
        )

        for subject in target_subjects

    ]

)


display(
    validation_table
)


if not all(

    (
        validation_table[
            [
                "class_0",
                "class_1",
                "class_2",
                "class_3"
            ]
        ]
        > 0
    ).all(
        axis=1
    )

):

    raise RuntimeError(
        "At least one target subject "
        "is missing a class."
    )


print(
    "✓ All selected target subjects "
    "contain all four classes."
)

,subject,trials,class_0,class_1,class_2,class_3,runs
0,4,90,23,22,22,23,"{4: 15, 6: 15, 8: 15, 10: 15, 12: 15, 14: 15}"
1,15,90,23,22,22,23,"{4: 15, 6: 15, 8: 15, 10: 15, 12: 15, 14: 15}"
2,23,90,22,23,23,22,"{4: 15, 6: 15, 8: 15, 10: 15, 12: 15, 14: 15}"
3,29,90,23,22,22,23,"{4: 15, 6: 15, 8: 15, 10: 15, 12: 15, 14: 15}"
4,31,90,22,23,23,22,"{4: 15, 6: 15, 8: 15, 10: 15, 12: 15, 14: 15}"
5,42,90,22,23,24,21,"{4: 15, 6: 15, 8: 15, 10: 15, 12: 15, 14: 15}"
6,55,90,23,22,23,22,"{4: 15, 6: 15, 8: 15, 10: 15, 12: 15, 14: 15}"
7,71,90,22,23,23,22,"{4: 15, 6: 15, 8: 15, 10: 15, 12: 15, 14: 15}"
8,82,90,23,22,24,21,"{4: 15, 6: 15, 8: 15, 10: 15, 12: 15, 14: 15}"
9,95,90,23,22,22,23,"{4: 15, 6: 15, 8: 15, 10: 15, 12: 15, 14: 15}"


✓ All selected target subjects contain all four classes.


In [11]:
# ============================================================
# CELL 11 - GRADIENT REVERSAL
# ============================================================

class GradientReversalLayer(
    torch.autograd.Function
):

    @staticmethod
    def forward(
        ctx,
        x,
        lambda_grl
    ):

        ctx.lambda_grl = lambda_grl

        return x.view_as(x)


    @staticmethod
    def backward(
        ctx,
        gradient
    ):

        return (

            -ctx.lambda_grl
            * gradient,

            None

        )


def grl(
    x,
    lambda_grl
):

    return GradientReversalLayer.apply(
        x,
        lambda_grl
    )

In [28]:
# ============================================================
# CELL 12 - FIXED STABLE LEARNABLE SINC FILTER BANK
# ============================================================

class SincFilterBank(nn.Module):

    def __init__(
        self,
        num_filters=NUM_FILTERS,
        kernel_size=SINC_KERNEL_SIZE,
        sample_rate=FS,
        min_freq=SINC_MIN_FREQ,
        max_freq=SINC_MAX_FREQ
    ):

        super().__init__()

        self.num_filters = num_filters
        self.kernel_size = kernel_size
        self.sample_rate = sample_rate

        self.min_freq = float(min_freq)
        self.max_freq = float(max_freq)

        # ----------------------------------------------------
        # Learnable low-frequency parameters
        # ----------------------------------------------------

        initial_low = torch.linspace(
            2.0,
            30.0,
            num_filters
        )

        self.low_parameter = nn.Parameter(
            initial_low
        )

        # ----------------------------------------------------
        # Learnable bandwidth
        # ----------------------------------------------------

        self.band_parameter = nn.Parameter(
            torch.ones(num_filters) * 5.0
        )


    def forward(self, x):

        B, C, T = x.shape

        # ----------------------------------------------------
        # Time axis
        # ----------------------------------------------------

        n = torch.arange(
            -(self.kernel_size // 2),
            self.kernel_size // 2 + 1,
            device=x.device,
            dtype=x.dtype
        )

        filters = []

        for filter_index in range(
            self.num_filters
        ):

            # ------------------------------------------------
            # Constrain LOW cutoff to 1-38 Hz
            # ------------------------------------------------

            low = torch.sigmoid(
                self.low_parameter[
                    filter_index
                ] / 10.0
            )

            low = (
                self.min_freq
                +
                low
                *
                (
                    self.max_freq
                    -
                    self.min_freq
                    -
                    2.0
                )
            )

            # ------------------------------------------------
            # Positive bandwidth
            # ------------------------------------------------

            bandwidth = F.softplus(
                self.band_parameter[
                    filter_index
                ]
            )

            # ------------------------------------------------
            # Calculate HIGH cutoff
            #
            # IMPORTANT FIX:
            # We first create a tensor upper bound.
            # ------------------------------------------------

            min_high = low + 1.0

            max_high = torch.tensor(
                self.max_freq,
                device=x.device,
                dtype=x.dtype
            )

            high = torch.minimum(
                min_high + bandwidth,
                max_high
            )

            # Safety protection
            high = torch.maximum(
                high,
                min_high
            )

            # ------------------------------------------------
            # Normalize frequencies
            # ------------------------------------------------

            f1 = low / self.sample_rate
            f2 = high / self.sample_rate

            # ------------------------------------------------
            # Sinc band-pass filter
            # ------------------------------------------------

            filter_kernel = (
                2.0
                * f2
                * torch.sinc(
                    2.0 * f2 * n
                )
                -
                2.0
                * f1
                * torch.sinc(
                    2.0 * f1 * n
                )
            )

            # ------------------------------------------------
            # Hamming window
            # ------------------------------------------------

            window = torch.hamming_window(
                self.kernel_size,
                device=x.device,
                dtype=x.dtype
            )

            filter_kernel = (
                filter_kernel * window
            )

            # ------------------------------------------------
            # Normalize filter energy
            # ------------------------------------------------

            filter_kernel = (
                filter_kernel
                /
                (
                    torch.sqrt(
                        torch.sum(
                            filter_kernel ** 2
                        )
                    )
                    + 1e-8
                )
            )

            filters.append(
                filter_kernel.view(
                    1,
                    1,
                    -1
                )
            )

        # ----------------------------------------------------
        # Stack filters
        # ----------------------------------------------------

        filters = torch.cat(
            filters,
            dim=0
        )

        # ----------------------------------------------------
        # Apply independently to each EEG channel
        # ----------------------------------------------------

        x_reshaped = x.reshape(
            B * C,
            1,
            T
        )

        output = F.conv1d(
            x_reshaped,
            filters,
            padding="same"
        )

        # ----------------------------------------------------
        # Restore dimensions
        #
        # B × F × C × T
        # ----------------------------------------------------

        output = output.reshape(
            B,
            C,
            self.num_filters,
            T
        )

        return output.permute(
            0,
            2,
            1,
            3
        )

In [29]:
# ============================================================
# CELL 13 - DYNAMIC GRAPH NEURAL NETWORK
# ============================================================

class DGNN(
    nn.Module
):

    def __init__(
        self,
        num_filters=NUM_FILTERS,
        in_nodes=N_CHANNELS,
        out_nodes=SPATIAL_DIM
    ):

        super().__init__()


        self.W_Q = nn.Linear(

            num_filters,

            num_filters

        )


        self.W_K = nn.Linear(

            num_filters,

            num_filters

        )


        self.W_V = nn.Linear(

            in_nodes,

            out_nodes

        )


        self.num_filters = \
            num_filters


    def forward(
        self,
        x
    ):

        B, F_bands, C, T = \
            x.shape


        # B x C x F

        descriptors = x.mean(
            dim=-1
        ).transpose(
            1,
            2
        )


        Q = self.W_Q(
            descriptors
        )


        K = self.W_K(
            descriptors
        )


        # ----------------------------------------------------
        # Dynamic adjacency
        # ----------------------------------------------------

        A = torch.matmul(

            Q,

            K.transpose(
                -2,
                -1
            )

        ) / math.sqrt(
            self.num_filters
        )


        A = F.softmax(
            A,
            dim=-1
        )


        # ----------------------------------------------------
        # Self connections
        # ----------------------------------------------------

        I = torch.eye(

            C,

            device=x.device,

            dtype=x.dtype

        ).unsqueeze(
            0
        )


        A_hat = A + I


        # ----------------------------------------------------
        # Symmetric normalization
        # ----------------------------------------------------

        degree = torch.clamp(

            A_hat.sum(
                dim=-1
            ),

            min=1e-6

        )


        D_inv_sqrt = \
            torch.diag_embed(

                1.0
                /
                torch.sqrt(
                    degree
                )

            )


        normalized_A = (

            D_inv_sqrt

            @

            A_hat

            @

            D_inv_sqrt

        )


        # ----------------------------------------------------
        # Graph convolution
        # ----------------------------------------------------

        x_transformed = \
            x.permute(
                0,
                1,
                3,
                2
            )


        output = torch.einsum(

            "bij,bntj->bnti",

            normalized_A,

            x_transformed

        )


        output = F.elu(

            self.W_V(
                output
            )

        )


        return output.permute(

            0,
            1,
            3,
            2

        )

In [30]:
# ============================================================
# CELL 14 - ENHANCED BIDIRECTIONAL TEMPORAL ENCODER
# ============================================================

class SimplifiedBiMamba(
    nn.Module
):

    def __init__(
        self,
        d_model=SPATIAL_DIM
    ):

        super().__init__()


        # ----------------------------------------------------
        # Larger 2-layer bidirectional GRU
        # ----------------------------------------------------

        self.gru = nn.GRU(

            input_size=d_model,

            hidden_size=GRU_HIDDEN,

            num_layers=GRU_LAYERS,

            dropout=GRU_DROPOUT,

            batch_first=True,

            bidirectional=True

        )


        # 128 -> 64
        self.projection = nn.Linear(

            GRU_HIDDEN * 2,

            d_model

        )


    def forward(
        self,
        x
    ):

        # B x D x T
        x = x.transpose(
            1,
            2
        )

        # B x T x 128
        output, _ = self.gru(
            x
        )

        # B x T x 64
        output = self.projection(
            output
        )

        # B x 64 x T
        return output.transpose(
            1,
            2
        )

In [31]:
# ============================================================
# CELL 15 - SE ATTENTION
# ============================================================

class SEAttention(
    nn.Module
):

    def __init__(
        self,
        channel=SPATIAL_DIM,
        reduction=16
    ):

        super().__init__()


        self.fc = nn.Sequential(

            nn.Linear(

                channel,

                channel // reduction,

                bias=False

            ),

            nn.ReLU(
                inplace=True
            ),

            nn.Linear(

                channel // reduction,

                channel,

                bias=False

            ),

            nn.Sigmoid()

        )


    def forward(
        self,
        x
    ):

        batch, channels, _ = \
            x.size()


        descriptor = x.mean(
            dim=2
        )


        attention = self.fc(
            descriptor
        ).view(

            batch,

            channels,

            1

        )


        weighted = (

            x
            *
            attention

        )


        return weighted.mean(
            dim=2
        )

In [32]:
# ============================================================
# CELL 16 - COMPLETE V2 MODEL
# ============================================================

class S3MambaDA(
    nn.Module
):

    def __init__(
        self,
        num_classes=NUM_CLASSES,
        num_subjects=TOTAL_SUBJECTS
    ):

        super().__init__()


        # ----------------------------------------------------
        # Spectral
        # ----------------------------------------------------

        self.sinc_filter = \
            SincFilterBank()


        # ----------------------------------------------------
        # Spatial
        # ----------------------------------------------------

        self.dgnn = DGNN()


        # ----------------------------------------------------
        # Temporal
        # ----------------------------------------------------

        self.temporal = \
            SimplifiedBiMamba()


        # ----------------------------------------------------
        # Attention
        # ----------------------------------------------------

        self.se_attention = \
            SEAttention()


        # ----------------------------------------------------
        # Classification
        # ----------------------------------------------------

        self.classifier = nn.Sequential(

            nn.BatchNorm1d(
                SPATIAL_DIM
            ),

            nn.Dropout(
                0.20
            ),

            nn.Linear(

                SPATIAL_DIM,

                num_classes

            )

        )


        # ----------------------------------------------------
        # Domain classifier
        # ----------------------------------------------------

        self.domain_classifier = \
            nn.Sequential(

                nn.Linear(
                    SPATIAL_DIM,
                    64
                ),

                nn.ReLU(),

                nn.Dropout(
                    0.20
                ),

                nn.Linear(
                    64,
                    num_subjects
                )

            )


        # ----------------------------------------------------
        # SupCon projection
        # ----------------------------------------------------

        self.supcon_projection = \
            nn.Sequential(

                nn.Linear(
                    SPATIAL_DIM,
                    128
                ),

                nn.ReLU(),

                nn.Linear(
                    128,
                    128
                )

            )


    def forward(
        self,
        x,
        lambda_grl=0.0
    ):

        # B × F × C × T
        spectral = self.sinc_filter(
            x
        )


        # B × F × D × T
        spatial = self.dgnn(
            spectral
        )


        # Frequency fusion
        temporal_input = \
            spatial.mean(
                dim=1
            )


        # Temporal encoder
        temporal = \
            self.temporal(
                temporal_input
            )


        # 64-D latent
        z = self.se_attention(
            temporal
        )


        # Classification
        class_logits = \
            self.classifier(
                z
            )


        # Domain branch
        reversed_z = grl(
            z,
            lambda_grl
        )


        domain_logits = \
            self.domain_classifier(
                reversed_z
            )


        # SupCon projection
        z_proj = F.normalize(

            self.supcon_projection(
                z
            ),

            p=2,

            dim=1

        )


        return (

            class_logits,

            domain_logits,

            z_proj

        )

In [33]:
# ============================================================
# CELL 17 - SUPERVISED CONTRASTIVE LOSS
# ============================================================

class SupConLoss(
    nn.Module
):

    def __init__(
        self,
        temperature=SUPCON_TEMPERATURE
    ):

        super().__init__()

        self.temperature = \
            temperature


    def forward(
        self,
        features,
        labels
    ):

        device = features.device


        similarity = (

            features
            @
            features.T

        ) / self.temperature


        labels = labels.view(
            -1,
            1
        )


        positive_mask = (

            labels
            ==
            labels.T

        ).float()


        # Remove self-comparisons
        logits_mask = torch.ones_like(
            positive_mask
        )


        logits_mask.fill_diagonal_(
            0
        )


        positive_mask *= logits_mask


        exp_logits = (

            torch.exp(
                similarity
            )
            *
            logits_mask

        )


        log_probability = (

            similarity

            -

            torch.log(

                exp_logits.sum(
                    dim=1,
                    keepdim=True
                )

                + 1e-8

            )

        )


        positive_count = \
            positive_mask.sum(
                dim=1
            )


        valid = (
            positive_count > 0
        )


        if not valid.any():

            return torch.zeros(

                (),

                device=device,

                requires_grad=True

            )


        mean_positive_log_probability = (

            positive_mask
            *
            log_probability

        ).sum(
            dim=1
        ) / (

            positive_count
            + 1e-8

        )


        return -mean_positive_log_probability[
            valid
        ].mean()

In [34]:
# ============================================================
# CELL 18 - MODEL SMOKE TEST
# ============================================================

model = S3MambaDA(
    num_classes=NUM_CLASSES,
    num_subjects=TOTAL_SUBJECTS
).to(DEVICE)


dummy = torch.randn(

    4,

    N_CHANNELS,

    N_SAMPLES,

    device=DEVICE

)


with torch.no_grad():

    class_logits, \
    domain_logits, \
    projection = model(

        dummy,

        lambda_grl=0.0

    )


print(
    "Input:",
    dummy.shape
)

print(
    "Class output:",
    class_logits.shape
)

print(
    "Domain output:",
    domain_logits.shape
)

print(
    "Projection:",
    projection.shape
)


parameter_count = sum(

    parameter.numel()

    for parameter in model.parameters()

)


print(
    "Parameters:",
    f"{parameter_count:,}"
)

Input: torch.Size([4, 22, 750])
Class output: torch.Size([4, 4])
Domain output: torch.Size([4, 109])
Projection: torch.Size([4, 128])
Parameters: 171,361


In [35]:
# ============================================================
# SINC FILTER SANITY TEST
# ============================================================

test_sinc = SincFilterBank().to(DEVICE)

test_x = torch.randn(
    2,
    N_CHANNELS,
    N_SAMPLES,
    device=DEVICE
)

with torch.no_grad():

    test_output = test_sinc(
        test_x
    )

print(
    "Input shape:",
    test_x.shape
)

print(
    "Sinc output shape:",
    test_output.shape
)

assert test_output.shape == (
    2,
    NUM_FILTERS,
    N_CHANNELS,
    N_SAMPLES
)

print(
    "✓ Sinc filter test passed."
)

Input shape: torch.Size([2, 22, 750])
Sinc output shape: torch.Size([2, 10, 22, 750])
✓ Sinc filter test passed.


In [36]:
# ============================================================
# CELL 19 - EVALUATION MODE
# ============================================================

def select_test_subjects():

    if EVAL_MODE == "PROJECT":

        return [

            subject

            for subject
            in CURATED_TEST_POOL

            if subject
            in AVAILABLE_SUBJECTS

        ][
            :NUM_TEST_FOLDS
        ]


    if EVAL_MODE == "RANDOM":

        generator = random.Random(
            SEED
        )

        return generator.sample(

            AVAILABLE_SUBJECTS,

            min(
                NUM_TEST_FOLDS,
                len(
                    AVAILABLE_SUBJECTS
                )
            )

        )


    if EVAL_MODE == "EXHAUSTIVE":

        return list(
            AVAILABLE_SUBJECTS
        )


    raise ValueError(
        "Invalid EVAL_MODE"
    )


TEST_SUBJECTS = \
    select_test_subjects()


print(
    "Evaluation mode:",
    EVAL_MODE
)

print(
    "Test subjects:",
    TEST_SUBJECTS
)

Evaluation mode: PROJECT
Test subjects: [4, 15, 23, 29, 31, 42, 55, 71, 82, 95]


In [37]:
# ============================================================
# CELL 20 - SOURCE TRAIN / VALIDATION SPLIT
# ============================================================

def split_source_subjects(
    source_subjects,
    validation_fraction=
        VALIDATION_SUBJECT_FRACTION,
    seed=SEED
):

    if len(
        source_subjects
    ) < 5:

        raise ValueError(
            "Too few source subjects."
        )


    validation_count = max(

        1,

        int(
            len(source_subjects)
            *
            validation_fraction
        )

    )


    generator = np.random.default_rng(
        seed
    )


    shuffled = generator.permutation(
        source_subjects
    )


    validation_subjects = \
        shuffled[
            :validation_count
        ].tolist()


    train_subjects = \
        shuffled[
            validation_count:
        ].tolist()


    return (
        train_subjects,
        validation_subjects
    )

In [38]:
# ============================================================
# CELL 21 - VALIDATION EVALUATOR
# ============================================================

def evaluate_model(
    model,
    loader
):

    model.eval()


    labels = []

    predictions = []


    with torch.no_grad():

        for x, y, _ in loader:

            x = x.to(
                DEVICE
            )

            logits, _, _ = model(
                x,
                lambda_grl=0.0
            )


            pred = torch.argmax(
                logits,
                dim=1
            )


            labels.extend(
                y.numpy()
            )

            predictions.extend(
                pred.cpu().numpy()
            )


    accuracy = accuracy_score(
        labels,
        predictions
    )


    report = classification_report(

        labels,

        predictions,

        labels=list(
            range(NUM_CLASSES)
        ),

        target_names=CLASS_NAMES,

        output_dict=True,

        zero_division=0

    )


    return {

        "accuracy":
            accuracy,

        "macro_f1":
            report[
                "macro avg"
            ][
                "f1-score"
            ]

    }

In [39]:
# ============================================================
# CELL 22 - ADAPTIVE BATCH NORMALIZATION
# ============================================================

def apply_adabn(

    model,

    target_loader,

    adaptation_trials

):

    model.eval()


    target_batches = []

    collected = 0


    for x, _, _ in target_loader:

        target_batches.append(
            x
        )

        collected += x.size(0)


        if collected >= \
           adaptation_trials:

            break


    if not target_batches:

        return model


    target_x = torch.cat(

        target_batches,

        dim=0

    )[

        :adaptation_trials

    ].to(
        DEVICE
    )


    bn_layers = [

        module

        for module in model.modules()

        if isinstance(

            module,

            nn.modules.batchnorm._BatchNorm

        )

    ]


    if not bn_layers:

        return model


    for module in bn_layers:

        module.reset_running_stats()

        module.momentum = 1.0

        module.train()


    with torch.no_grad():

        _ = model(

            target_x,

            lambda_grl=0.0

        )


    for module in bn_layers:

        module.momentum = 0.1

        module.eval()


    model.eval()


    return model

In [40]:
# ============================================================
# CELL 23 - GRL / SUPCON WEIGHT SCHEDULE
# ============================================================

def auxiliary_weights(
    epoch
):

    # --------------------------------------------------------
    # Warm-up
    # --------------------------------------------------------

    if epoch < WARMUP_EPOCHS:

        return (
            0.0,
            0.0
        )


    # --------------------------------------------------------
    # Gradual increase
    # --------------------------------------------------------

    progress = (

        epoch
        -
        WARMUP_EPOCHS

    ) / max(

        1,

        EPOCHS
        -
        WARMUP_EPOCHS

    )


    progress = float(
        np.clip(
            progress,
            0.0,
            1.0
        )
    )


    # Smooth sigmoid-like ramp
    ramp = (

        2.0
        /
        (
            1.0
            +
            np.exp(
                -10.0
                *
                (
                    progress
                    -
                    0.5
                )
            )
        )

    )


    domain_weight = (

        DOMAIN_WEIGHT_MAX
        *
        ramp

    )


    supcon_weight = (

        SUPCON_WEIGHT_MAX
        *
        ramp

    )


    return (
        domain_weight,
        supcon_weight
    )

In [41]:
# ============================================================
# CELL 24 - TRAIN ONE FOLD
# ============================================================

def train_fold(

    model,

    train_loader,

    validation_loader

):

    criterion_classification = \
        nn.CrossEntropyLoss(

            label_smoothing=
                LABEL_SMOOTHING

        )


    criterion_domain = \
        nn.CrossEntropyLoss()


    criterion_supcon = \
        SupConLoss()


    optimizer = \
        torch.optim.AdamW(

            model.parameters(),

            lr=LEARNING_RATE,

            weight_decay=
                WEIGHT_DECAY

        )


    scheduler = \
        torch.optim.lr_scheduler.CosineAnnealingLR(

            optimizer,

            T_max=EPOCHS,

            eta_min=1e-5

        )


    best_validation_f1 = -np.inf

    best_epoch = 0

    best_state = None


    history = []


    for epoch in range(
        EPOCHS
    ):

        model.train()


        epoch_total = 0.0

        epoch_classification = 0.0

        epoch_domain = 0.0

        epoch_supcon = 0.0

        count = 0


        domain_weight, \
        supcon_weight = \
            auxiliary_weights(
                epoch
            )


        # ----------------------------------------------------
        # TRAINING
        # ----------------------------------------------------

        for x, y, subject in train_loader:

            x = x.to(
                DEVICE
            )

            y = y.to(
                DEVICE
            )

            subject = subject.to(
                DEVICE
            )


            optimizer.zero_grad(
                set_to_none=True
            )


            # ------------------------------------------------
            # GRL schedule
            # ------------------------------------------------

            if epoch < WARMUP_EPOCHS:

                lambda_grl = 0.0

            else:

                p = (

                    epoch
                    -
                    WARMUP_EPOCHS

                ) / max(

                    1,

                    EPOCHS
                    -
                    WARMUP_EPOCHS

                )


                lambda_grl = (

                    2.0
                    /
                    (
                        1.0
                        +
                        np.exp(
                            -10.0
                            * p
                        )
                    )

                    - 1.0

                )


                lambda_grl *= (
                    domain_weight
                    /
                    max(
                        DOMAIN_WEIGHT_MAX,
                        1e-8
                    )
                )


            # ------------------------------------------------
            # Forward
            # ------------------------------------------------

            logits, \
            domain_logits, \
            z_projection = model(

                x,

                lambda_grl=
                    lambda_grl

            )


            loss_classification = \
                criterion_classification(

                    logits,

                    y

                )


            loss_domain = \
                criterion_domain(

                    domain_logits,

                    subject

                )


            loss_supcon = \
                criterion_supcon(

                    z_projection,

                    y

                )


            # ------------------------------------------------
            # Total loss
            # ------------------------------------------------

            loss_total = (

                loss_classification

                +

                domain_weight
                *
                loss_domain

                +

                supcon_weight
                *
                loss_supcon

            )


            loss_total.backward()


            torch.nn.utils.clip_grad_norm_(

                model.parameters(),

                max_norm=
                    GRADIENT_CLIP

            )


            optimizer.step()


            batch_size = x.size(0)

            count += batch_size


            epoch_total += (

                loss_total.detach()
                .cpu()
                .item()
                *
                batch_size

            )


            epoch_classification += (

                loss_classification.detach()
                .cpu()
                .item()
                *
                batch_size

            )


            epoch_domain += (

                loss_domain.detach()
                .cpu()
                .item()
                *
                batch_size

            )


            epoch_supcon += (

                loss_supcon.detach()
                .cpu()
                .item()
                *
                batch_size

            )


        scheduler.step()


        # ----------------------------------------------------
        # Validation
        # ----------------------------------------------------

        validation_metrics = \
            evaluate_model(

                model,

                validation_loader

            )


        average_total = (
            epoch_total
            /
            max(
                count,
                1
            )
        )


        average_classification = (
            epoch_classification
            /
            max(
                count,
                1
            )
        )


        average_domain = (
            epoch_domain
            /
            max(
                count,
                1
            )
        )


        average_supcon = (
            epoch_supcon
            /
            max(
                count,
                1
            )
        )


        record = {

            "epoch":
                epoch + 1,

            "total_loss":
                average_total,

            "classification_loss":
                average_classification,

            "domain_loss":
                average_domain,

            "supcon_loss":
                average_supcon,

            "domain_weight":
                domain_weight,

            "supcon_weight":
                supcon_weight,

            "lambda_grl":
                lambda_grl,

            "validation_accuracy":
                validation_metrics[
                    "accuracy"
                ],

            "validation_macro_f1":
                validation_metrics[
                    "macro_f1"
                ],

            "learning_rate":
                optimizer.param_groups[
                    0
                ]["lr"]

        }


        history.append(
            record
        )


        # ----------------------------------------------------
        # Best model
        # ----------------------------------------------------

        validation_f1 = \
            validation_metrics[
                "macro_f1"
            ]


        if validation_f1 > \
           best_validation_f1:

            best_validation_f1 = \
                validation_f1

            best_epoch = \
                epoch + 1

            best_state = copy.deepcopy(
                model.state_dict()
            )


        # ----------------------------------------------------
        # Logging
        # ----------------------------------------------------

        if (

            epoch == 0
            or
            (epoch + 1) % 10 == 0

        ):

            print(

                f"Epoch "
                f"{epoch+1:03d}/{EPOCHS} | "
                f"Total={average_total:.4f} | "
                f"Cls={average_classification:.4f} | "
                f"Domain={average_domain:.4f} | "
                f"SupCon={average_supcon:.4f} | "
                f"ValF1={validation_f1:.4f}"

            )


    # --------------------------------------------------------
    # Restore best model
    # --------------------------------------------------------

    if best_state is not None:

        model.load_state_dict(
            best_state
        )


    return (

        model,

        pd.DataFrame(
            history
        ),

        best_epoch,

        best_validation_f1

    )

In [42]:
# ============================================================
# CELL 25 - FINAL TEST EVALUATOR
# ============================================================

def final_test(

    model,

    test_loader

):

    model.eval()


    labels = []

    predictions = []

    probabilities = []

    embeddings = []


    with torch.no_grad():

        for x, y, _ in test_loader:

            x = x.to(
                DEVICE
            )


            logits, _, z = model(

                x,

                lambda_grl=0.0

            )


            probs = torch.softmax(

                logits,

                dim=1

            )


            preds = torch.argmax(

                probs,

                dim=1

            )


            labels.extend(
                y.numpy()
            )


            predictions.extend(
                preds.cpu().numpy()
            )


            probabilities.append(
                probs.cpu().numpy()
            )


            embeddings.append(
                z.cpu().numpy()
            )


    probabilities = \
        np.concatenate(
            probabilities,
            axis=0
        )


    embeddings = \
        np.concatenate(
            embeddings,
            axis=0
        )


    accuracy = \
        accuracy_score(
            labels,
            predictions
        )


    kappa = \
        cohen_kappa_score(
            labels,
            predictions
        )


    report = \
        classification_report(

            labels,

            predictions,

            labels=[
                0,
                1,
                2,
                3
            ],

            target_names=
                CLASS_NAMES,

            output_dict=True,

            zero_division=0

        )


    metrics = {

        "accuracy":
            accuracy,

        "kappa":
            kappa,

        "macro_precision":
            report[
                "macro avg"
            ][
                "precision"
            ],

        "macro_recall":
            report[
                "macro avg"
            ][
                "recall"
            ],

        "macro_f1":
            report[
                "macro avg"
            ][
                "f1-score"
            ]

    }


    return (

        metrics,

        np.array(
            labels
        ),

        np.array(
            predictions
        ),

        probabilities,

        embeddings

    )

In [ ]:
# ============================================================
# CELL 27 - RUN V2 EXPERIMENT
# ============================================================

fold_df, \
predictions_df, \
embeddings_df, \
history_df, \
summary = run_experiment()


S3MambaDA V2 EXPERIMENT
Evaluation mode: PROJECT
Test subjects: [4, 15, 23, 29, 31, 42, 55, 71, 82, 95]
Input: 22 × 750
Epochs: 200
Warm-up: 20
LR: 0.0003

##########################################################################################
FOLD 1/10
TARGET: S004
##########################################################################################
Source subjects: 99
Train subjects: 90
Validation subjects: 9
Training trials: 8128
Validation trials: 810
Test trials: 90
Target distribution: Counter({0: 23, 3: 23, 1: 22, 2: 22})
Epoch 001/200 | Total=1.4027 | Cls=1.4027 | Domain=4.6937 | SupCon=4.1431 | ValF1=0.1594


##### ============================================================
# CELL 29 - FINAL STATISTICS
# ============================================================

print(
    f"Mean Accuracy: "
    f"{summary['mean_accuracy']*100:.2f}%"
)

print(
    f"Accuracy SD: "
    f"{summary['std_accuracy']*100:.2f}%"
)

print(
    f"Mean Kappa: "
    f"{summary['mean_kappa']:.4f}"
)

print(
    f"Mean Macro F1: "
    f"{summary['mean_macro_f1']:.4f}"
)

print(
    f"Total test samples: "
    f"{summary['total_test_samples']}"
)